# Seminar 15: Diffusion Playground - Prompts, Seeds, Guidance, and Editing

**Student Version**

Today is a controlled diffusion lab with a creative challenge at the end.

Goals:
- generate a first image with a pretrained diffusion model,
- compare prompt variants while keeping the seed fixed,
- compare seeds while keeping the prompt fixed,
- test guidance scale as a prompt-following knob,
- test negative prompts without treating them as magic,
- try an editing task with image-to-image generation,
- run a mini showcase where every team explains its settings and iterations.

Expected rhythm: first generation -> prompt experiments -> seed experiments -> guidance scale -> negative prompts -> editing task -> mini showcase.

## 0. Setup

Recommended: **Google Colab with GPU**.

If package installation or model loading fails, use the fallback path from the instructor. The seminar is about controlled generation experiments, not about fighting environment setup.

In [ ]:
# If needed in Colab, uncomment:
# !pip install -q diffusers transformers accelerate safetensors

import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from PIL import Image

from diffusers import AutoPipelineForText2Image, AutoPipelineForImage2Image

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = torch.float16 if device == 'cuda' else torch.float32
device, dtype


### Config

The default checkpoint is chosen for a Colab-style diffusion lab. If it is unavailable, your instructor may switch to a different model or hosted tool.

In [ ]:
CFG = {
    'model_id': 'runwayml/stable-diffusion-v1-5',
    'height': 512,
    'width': 512,
    'num_inference_steps': 25,
    'default_guidance': 7.5,
    'output_dir': 'seminar15_outputs',
}

Path(CFG['output_dir']).mkdir(exist_ok=True)
CFG


### Load the pipeline

This is setup, not an exercise. The first run may take a few minutes.

In [ ]:
if device != 'cuda':
    print('Warning: no GPU detected. Generation will be slow.')

pipe = AutoPipelineForText2Image.from_pretrained(
    CFG['model_id'],
    torch_dtype=dtype,
)
pipe = pipe.to(device)

if hasattr(pipe, 'enable_attention_slicing'):
    pipe.enable_attention_slicing()

pipe


### Helper functions

These helpers are provided. Your exercises will change prompts, seeds, guidance, and negative prompts.

In [ ]:
experiment_log = []

def make_generator(seed: int):
    return torch.Generator(device=device).manual_seed(seed)

def safe_name(text: str, max_len: int = 42):
    keep = []
    for ch in text.lower().replace(' ', '_'):
        keep.append(ch if ch.isalnum() or ch in ['_', '-'] else '')
    return ''.join(keep)[:max_len] or 'image'

def generate_image(
    prompt: str,
    seed: int = 42,
    guidance_scale: float = None,
    negative_prompt: str = '',
    steps: int = None,
    label: str = 'experiment',
):
    guidance_scale = CFG['default_guidance'] if guidance_scale is None else guidance_scale
    steps = CFG['num_inference_steps'] if steps is None else steps

    result = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt or None,
        generator=make_generator(seed),
        guidance_scale=guidance_scale,
        num_inference_steps=steps,
        height=CFG['height'],
        width=CFG['width'],
    )
    image = result.images[0]
    filename = Path(CFG['output_dir']) / f"{len(experiment_log):03d}_{safe_name(label)}_seed{seed}_g{guidance_scale}.png"
    image.save(filename)

    experiment_log.append({
        'label': label,
        'prompt': prompt,
        'negative_prompt': negative_prompt,
        'seed': seed,
        'guidance_scale': guidance_scale,
        'steps': steps,
        'file': str(filename),
        'observation': '',
    })
    return image

def show_images(images, titles=None, cols=4, figsize=(14, 7)):
    titles = titles or [''] * len(images)
    rows = int(np.ceil(len(images) / cols))
    plt.figure(figsize=figsize)
    for i, image in enumerate(images):
        ax = plt.subplot(rows, cols, i + 1)
        ax.imshow(image)
        ax.set_title(titles[i], fontsize=10)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

def show_log(last_n=10):
    return pd.DataFrame(experiment_log).tail(last_n)


## 1. First Generation

Task:
- choose a prompt,
- choose a seed,
- choose a guidance scale,
- generate one image,
- inspect whether it follows the prompt.

Function contract:
- `first_prompt` should be a string,
- `first_seed` should be an integer,
- `first_guidance` should be a float.

In [ ]:
# Exercise 1.1

# TODO: choose a prompt, seed, and guidance scale
# first_prompt = ...
# first_seed = ...
# first_guidance = ...

raise NotImplementedError('Fill in Exercise 1.1')

first_image = generate_image(
    first_prompt,
    seed=first_seed,
    guidance_scale=first_guidance,
    label='first_generation',
)
first_image


### Checks (First Generation)

1. What is good about the image?
2. What is wrong or missing?
3. Which prompt words seem visible in the image?

## 2. Prompt Experiments

Keep the seed and guidance scale fixed. Change only prompt wording.

Task:
- create four prompt variants,
- keep the same seed and guidance scale for all variants,
- compare the outputs.

Expected prompt types:
1. short prompt,
2. detailed prompt,
3. photo/documentary style prompt,
4. illustration/anime/art style prompt.

In [ ]:
# Exercise 2.1

# TODO: write four prompt variants around the same main subject
# prompt_variants = [
#     ...,
#     ...,
#     ...,
#     ...,
# ]

raise NotImplementedError('Fill in Exercise 2.1')

prompt_seed = 10
prompt_guidance = 7.5

prompt_images = [
    generate_image(prompt, seed=prompt_seed, guidance_scale=prompt_guidance, label=f'prompt_variant_{i}')
    for i, prompt in enumerate(prompt_variants)
]
show_images(prompt_images, [f'Prompt {i}' for i in range(len(prompt_images))], cols=2)


### Checks (Prompt Experiments)

1. Which words controlled the subject?
2. Which words controlled style or camera feel?
3. Did adding details always improve the image?

## 3. Seed Experiments

Keep the prompt and guidance scale fixed. Change only the seed.

Main idea:

```text
prompt controls direction
seed controls one random starting point
```

Task:
- choose one prompt,
- choose four different seeds,
- compare what stays stable and what changes.

In [ ]:
# Exercise 3.1

# TODO: choose one prompt and four integer seeds
# seed_prompt = ...
# seeds = [..., ..., ..., ...]

raise NotImplementedError('Fill in Exercise 3.1')

seed_images = [
    generate_image(seed_prompt, seed=seed, guidance_scale=7.5, label=f'seed_{seed}')
    for seed in seeds
]
show_images(seed_images, [f'seed={seed}' for seed in seeds], cols=4)


### Checks (Seed Experiments)

1. What stayed stable across seeds?
2. What changed across seeds?
3. Why is saving the seed useful for reproducibility?

## 4. Guidance Scale

Keep prompt and seed fixed. Change only guidance scale.

Task:
- choose one prompt,
- keep one seed fixed,
- compare at least four guidance values,
- decide whether higher guidance is actually better.

In [ ]:
# Exercise 4.1

# TODO: choose one prompt, one seed, and at least four guidance values
# guidance_prompt = ...
# guidance_seed = ...
# guidance_values = [..., ..., ..., ...]

raise NotImplementedError('Fill in Exercise 4.1')

guidance_images = [
    generate_image(guidance_prompt, seed=guidance_seed, guidance_scale=value, label=f'guidance_{value}')
    for value in guidance_values
]
show_images(guidance_images, [f'guidance={value}' for value in guidance_values], cols=4)


### Checks (Guidance Scale)

1. Which guidance value followed the prompt best?
2. Which guidance value looked most natural?
3. Did high guidance create any artifacts?

## 5. Negative Prompts

Negative prompts steer the model away from some concepts.

They are useful, but they are not hard guarantees.

Task:
- generate one image without a negative prompt,
- generate the same prompt/seed/guidance with a negative prompt,
- compare the outputs.

In [ ]:
# Exercise 5.1

# TODO: choose a prompt and negative prompt
# negative_test_prompt = ...
# negative_prompt = ...

raise NotImplementedError('Fill in Exercise 5.1')

negative_seed = 99
negative_guidance = 7.5

without_negative = generate_image(
    negative_test_prompt,
    seed=negative_seed,
    guidance_scale=negative_guidance,
    negative_prompt='',
    label='without_negative',
)
with_negative = generate_image(
    negative_test_prompt,
    seed=negative_seed,
    guidance_scale=negative_guidance,
    negative_prompt=negative_prompt,
    label='with_negative',
)

show_images([without_negative, with_negative], ['no negative prompt', 'with negative prompt'], cols=2)


### Checks (Negative Prompts)

1. Did the negative prompt fix a real problem?
2. Did it introduce a different problem?
3. Would you use the same negative prompt for every image?

## 6. Editing Task

Image-to-image generation starts from an existing image and changes it according to a new prompt.

Task:
- choose an image from earlier in the notebook,
- write an edit prompt,
- choose a strength value,
- compare the original and edited image.

If this section is slow or unstable, skip it and continue to the challenge.

In [ ]:
# Exercise 6.1

# TODO: choose an image from earlier, an edit prompt, and strength value
# init_image = ...
# edit_prompt = ...
# edit_strength = ...

raise NotImplementedError('Fill in Exercise 6.1')

img2img_pipe = AutoPipelineForImage2Image.from_pipe(pipe)
init_image = init_image.resize((CFG['width'], CFG['height']))

edited_image = img2img_pipe(
    prompt=edit_prompt,
    image=init_image,
    strength=edit_strength,
    guidance_scale=7.5,
    num_inference_steps=CFG['num_inference_steps'],
    generator=make_generator(777),
).images[0]

show_images([init_image, edited_image], ['original', 'edited'], cols=2)


### Checks (Editing Task)

1. What did the edit preserve?
2. What changed?
3. Is this closer to image editing or generating a new image?

## 7. Mini Showcase: Diffusion Control Challenge

Work in teams. Create one final image and explain how you controlled the generation.

Choose one category:

1. **Best Prompt Following** - satisfy a complex prompt with multiple constraints.
2. **Best Style Control** - same subject, strong intentional visual style.
3. **Best Iteration Story** - show how changing seed/guidance/negative prompt improved the result.
4. **Funniest Failure** - find an interesting failure and explain why it may have happened.
5. **Most Useful Image** - something useful for a slide, poster, product mockup, or dataset idea.

Rule:

> Make at least three attempts. For at least two attempts, change only one variable at a time.

In [ ]:
# Exercise 7.1

# TODO: complete your team challenge variables
# team_category = ...
# team_prompt = ...
# team_negative_prompt = ...
# team_seed = ...
# team_guidance = ...

raise NotImplementedError('Fill in Exercise 7.1')

team_image = generate_image(
    team_prompt,
    seed=team_seed,
    guidance_scale=team_guidance,
    negative_prompt=team_negative_prompt,
    label='team_final_candidate',
)
team_image


In [ ]:
# Exercise 7.2

# TODO: fill in the submission summary before the showcase
submission = {
    'team_name': '',
    'category': '',
    'final_prompt': '',
    'negative_prompt': '',
    'seed': '',
    'guidance_scale': '',
    'number_of_attempts': '',
    'what_changed_during_iteration': '',
    'what_worked': '',
    'what_failed': '',
}

pd.Series(submission)


### Showcase scoring

| Criterion | Points |
|---|---:|
| Prompt/result match | 3 |
| Creativity | 3 |
| Technical explanation | 3 |
| Fun / surprise | 1 |

Total: **10 points**

A beautiful image with no explanation should not beat a controlled experiment with a clear story.

## 8. Wrap-Up Questions

1. Which knob gave you the most control?
2. Which knob was most unpredictable?
3. Did negative prompts behave like hard constraints?
4. What failure mode appeared most often?
5. What would you need to log if this were a real product feature?
6. How is this similar to the experiment discipline we used for classifiers?

## Optional Extension

Try one of these:
- use the same prompt and compare more seeds,
- try a very low and very high guidance value,
- generate a useful image for a class slide,
- create a failure gallery and classify failures by type,
- try another editing strength for image-to-image generation.